# Lab 2: Global Health Data Exploration

This notebook reads cached raw files from `data/raw/`. If a required file is missing, its extractor downloads it once and saves it for later runs. Run the notebook from the project root with the project environment active. The World Bank example is GDP per capita; WHO and OWID examples follow the Lab 2 indicator list.

In [ ]:
from pathlib import Path

import pandas as pd

from etl.extract_owid import extract_owid
from etl.extract_who import extract_who
from etl.extract_worldbank import extract_worldbank, extract_worldbank_countries
from etl.transform_owid import transform_owid
from etl.transform_who import transform_who
from etl.transform_worldbank import transform_worldbank

RAW_ROOT = Path("data/raw")
WORLD_BANK_INDICATOR = "NY.GDP.PCAP.CD"
WHO_INDICATOR = "WHOSIS_000001"
OWID_CHART = "life-expectancy"


## Read or download raw files

Each extractor returns the saved file path and request URL. Existing files are reused unless explicitly forced, so rerunning these cells does not repeatedly download the same data.

In [ ]:
worldbank_countries_file, worldbank_countries_url = extract_worldbank_countries(
    raw_dir=RAW_ROOT / "worldbank"
)
worldbank_file, worldbank_url = extract_worldbank(
    WORLD_BANK_INDICATOR,
    year_range=(2000, 2024),
    raw_dir=RAW_ROOT / "worldbank",
)
who_file, who_url = extract_who(
    WHO_INDICATOR,
    year_range=(2000, 2024),
    raw_dir=RAW_ROOT / "who",
)
owid_file, owid_url = extract_owid(
    OWID_CHART,
    raw_dir=RAW_ROOT / "owid",
)

raw_files = {
    "worldbank": (worldbank_file, worldbank_url),
    "who": (who_file, who_url),
    "owid": (owid_file, owid_url),
}
raw_files

## Transform to a common structure

The source-specific transforms keep extraction separate from normalization. All frames below contain `country_iso3`, `indicator_code`, `year`, `sex`, and `value`.

In [ ]:
worldbank_df = transform_worldbank(worldbank_file, worldbank_countries_file)
who_df = transform_who(who_file)
owid_df = transform_owid(owid_file, indicator_code="LIFE-EXPECTANCY")

df = pd.concat([worldbank_df, who_df, owid_df], ignore_index=True)
df.head()

The first rows provide a quick structural check. Differences in units or population definitions remain source-specific and should be considered before comparing indicator values.

In [ ]:
df.info()

## Missing-value inspection

Missing values are measured rather than filled with zero. The percentages below are calculated separately for each indicator and normalized field.

In [ ]:
df.isna().sum()

In [ ]:
normalized_columns = [
    "country_iso3",
    "indicator_code",
    "year",
    "sex",
    "value",
]
missing_counts = df.groupby("indicator_code")[normalized_columns].agg(
    lambda series: series.isna().sum()
)
indicator_row_counts = df.groupby("indicator_code").size()
missing_percent_by_indicator = (
    missing_counts.div(indicator_row_counts, axis="index").mul(100)
)
missing_percent_by_indicator

Review the resulting percentages before cleaning. Missingness can differ substantially by source and indicator. This exploratory view does not remove rows, impute values, or load anything into PostgreSQL.